# Clover Cancer — Gemma 4 Fine-tuning

Fine-tunes Google's Gemma 4 E2B model on pancreatic cancer triage data using Unsloth.

**Hardware:** Kaggle T4 GPU (16GB VRAM)  
**Author:** Adhyaay Karnwal  
**Competition:** [Gemma 4 Good Hackathon](https://www.kaggle.com/competitions/gemma-4-good-hackathon)

## Setup

In [ ]:
%%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops "flash-attn>=2.6.3"

In [ ]:
from unsloth import FastLanguageModel
import torch
import json
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Unsloth quantizes the model to 4-bit and attaches LoRA adapters so everything fits on the T4's 16 GB VRAM.

## Configuration

In [ ]:
MODEL_NAME = "unsloth/gemma-4-e2b-it"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"]

LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 8

## Load Model

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

# Verify LoRA attached correctly
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {all_params:,} ({100*trainable_params/all_params:.2f}%)")

## Load Dataset

Upload `train.json` and `val.json` from `data/processed/` as a Kaggle dataset.

In [ ]:
# Update this path to match your Kaggle dataset input
DATA_DIR = "/kaggle/input/datasets/adhyaaykarnwal/clover-cancer-data"

with open(os.path.join(DATA_DIR, "train.json")) as f:
    train_data = json.load(f)

with open(os.path.join(DATA_DIR, "val.json")) as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)} examples")
print(f"Val: {len(val_data)} examples")
print(f"\nExample conversation:")
print(json.dumps(train_data[0]["conversations"][0], indent=2)[:200])

In [ ]:
from datasets import Dataset

def format_conversations(examples):
    texts = []
    for convo in examples["conversations"]:
        # Gemma 4 expects 'model' not 'assistant' as the role
        # Also merge system message into user message if present
        messages = []
        system_content = None
        for msg in convo:
            if msg["role"] == "system":
                system_content = msg["content"]
            elif msg["role"] == "user":
                content = msg["content"]
                if system_content:
                    content = f"{system_content}\n\n{content}"
                    system_content = None
                messages.append({"role": "user", "content": content})
            elif msg["role"] == "assistant":
                messages.append({"role": "model", "content": msg["content"]})

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        ).removeprefix("<bos>")
        texts.append(text)
    return {"text": texts}

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

train_dataset = train_dataset.map(format_conversations, batched=True, remove_columns=train_dataset.column_names)
val_dataset = val_dataset.map(format_conversations, batched=True, remove_columns=val_dataset.column_names)

# Debug: print a sample to verify format
print("Sample formatted text (first 500 chars):")
print(train_dataset[0]["text"][:500])
print(f"\nContains '<start_of_turn>model\n': {'<start_of_turn>model' in train_dataset[0]['text']}")

print(f"\nTrain formatted: {len(train_dataset)} examples")
print(f"Val formatted: {len(val_dataset)} examples")

Set up the trainer using SFTTrainer from TRL. I use `train_on_responses_only` so the model learns from its own responses, not from the user input.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_ratio=0.05,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=5,
        save_strategy="epoch",
        eval_strategy="epoch",
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        max_seq_length=MAX_SEQ_LENGTH,
        max_grad_norm=0.3,
        fp16=True,
    ),
)

# Gemma 4 uses <|turn>user and <|turn>model markers
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|turn>user\n",
    response_part="<|turn>model\n",
)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Max memory: {max_memory} GB")
print(f"Starting memory: {start_gpu_memory} GB")

In [ ]:
stats = trainer.train()

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nTraining complete!")
print(f"  Steps: {stats.global_step}")
print(f"  Loss: {stats.training_loss:.4f}")
print(f"  Time: {stats.metrics['train_runtime']:.0f}s")
print(f"  Peak memory: {used_memory}/{max_memory} GB ({100*used_memory/max_memory:.1f}%)")

## Save Model

In [ ]:
# Save LoRA adapters (small, reliable)
model.save_pretrained("clover-cancer-lora")
tokenizer.save_pretrained("clover-cancer-lora")
print("LoRA adapters saved to clover-cancer-lora/")
print("Download this folder from the Output panel on the right.")

In [ ]:
TEST_SCENARIOS = [
    {
        "name": "Classic high-risk",
        "input": "I'm a 58-year-old male. I was just diagnosed with diabetes last month. I've lost about 15 pounds without trying. I have this deep ache in my mid-back that won't go away.",
        "expected": "high"
    },
    {
        "name": "Emergency jaundice",
        "input": "I'm a 67-year-old female. My skin turned yellow. I've lost about 10 pounds. My urine is really dark.",
        "expected": "critical"
    },
    {
        "name": "BRCA2 carrier",
        "input": "I'm a 52-year-old female with a BRCA2 mutation. My mother died of pancreatic cancer. I've been having abdominal pain and fatigue for 6 weeks.",
        "expected": "high"
    },
    {
        "name": "Low-risk back pain",
        "input": "I'm a 42-year-old male. I've had back pain for 2 weeks. I sit at a desk all day. No other symptoms.",
        "expected": "low"
    },
]

In [ ]:
SYSTEM_PROMPT = """You are a medical triage AI specialized in pancreatic cancer early detection.
Analyze symptoms and respond with JSON:
{"risk_assessment": "low|medium|high|critical", "conditions": [...], "urgency": "...", "recommended_actions": [...], "reasoning_chain": "..."}"""

FastLanguageModel.for_inference(model)

for scenario in TEST_SCENARIOS:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "text", "text": f"Assess my symptoms:\n\n{scenario['input']}"}]}
    ]

    input_ids = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=512,
        temperature=0.3,
        top_p=0.9,
        top_k=40,
    )

    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario['name']}")
    print(f"Expected risk: {scenario['expected']}")
    print(f"Response:\n{response}")

## Evaluation on 10 Clinical Scenarios (My custom eval benchmark)

Runs the fine-tuned model against 10 predefined test scenarios and computes risk accuracy, urgency accuracy, clinical term coverage, and reasoning quality scores.

In [ ]:
import re, json

EVAL_SCENARIOS = [
    {"name": "Classic high-risk pattern", "input": "I'm a 58-year-old male. I was just diagnosed with diabetes last month. I've lost about 15 pounds without trying. I have this deep ache in my mid-back that won't go away. This has been going on for about 3 months.", "expected_risk": "high", "expected_urgency": "urgent", "must_mention": ["pancreatic", "diabetes", "CT", "weight"]},
    {"name": "Emergency jaundice", "input": "I'm a 67-year-old female. My skin turned yellow and the whites of my eyes look yellow too. I've lost about 10 pounds in the last month. My urine is really dark, almost brown. I haven't had any pain really. This started about 2 weeks ago.", "expected_risk": "critical", "expected_urgency": "emergency", "must_mention": ["jaundice", "pancreatic", "urgent", "imaging"]},
    {"name": "BRCA2 carrier with symptoms", "input": "I'm a 52-year-old female. I have a BRCA2 mutation and my mother died of pancreatic cancer. I've been having abdominal pain and fatigue for about 6 weeks. I've lost some weight too, maybe 8 pounds.", "expected_risk": "high", "expected_urgency": "urgent", "must_mention": ["BRCA2", "pancreatic", "family history"]},
    {"name": "Low-risk back pain", "input": "I'm a 42-year-old male. I've had back pain and some stomach discomfort for about 2 weeks. I sit at a desk all day and haven't been exercising much. No weight loss or other symptoms.", "expected_risk": "low", "expected_urgency": "routine", "must_mention": ["musculoskeletal"], "should_not_mention": ["pancreatic cancer"]},
    {"name": "Steatorrhea pattern", "input": "I'm a 62-year-old female. My stools have been really greasy and foul-smelling. They float and are hard to flush. I've lost about 12 pounds in 3 months. I feel bloated after eating even small meals.", "expected_risk": "medium", "expected_urgency": "urgent", "must_mention": ["pancreatic", "imaging", "weight"]},
    {"name": "New-onset diabetes alone", "input": "I'm a 56-year-old male. My doctor just told me I have type 2 diabetes. My blood sugar has been running high. I don't have any other real symptoms, maybe a bit of stomach discomfort.", "expected_risk": "medium", "expected_urgency": "routine", "must_mention": ["diabetes", "pancreatic"]},
    {"name": "Recurrent pancreatitis", "input": "I'm a 58-year-old male. I've been to the ER three times this year for pancreatitis. Each time it's terrible pain in my upper abdomen that radiates to my back. I was also just diagnosed with diabetes. I've lost about 10 pounds.", "expected_risk": "high", "expected_urgency": "urgent", "must_mention": ["pancreatic", "pancreatitis", "CT"]},
    {"name": "DVT with abdominal symptoms", "input": "I'm a 63-year-old female. I've had two blood clots in my legs in the past year. I've also been having abdominal pain and lost about 15 pounds without trying. My doctor can't figure out why I keep getting clots.", "expected_risk": "high", "expected_urgency": "urgent", "must_mention": ["pancreatic", "Trousseau", "CT"]},
    {"name": "IBS in young patient", "input": "I'm a 34-year-old female. I've been having bloating, diarrhea, and some fatigue for about 2 months. It comes and goes. No weight loss, no family history of cancer.", "expected_risk": "low", "expected_urgency": "routine", "must_mention": ["IBS"], "should_not_mention": ["pancreatic cancer"]},
    {"name": "Mild jaundice (subtle)", "input": "I'm a 64-year-old male. I've been itching all over for weeks with no rash. My wife noticed my skin looks slightly yellowish. I've been more tired than usual. No pain. This has been going on about 3 weeks.", "expected_risk": "high", "expected_urgency": "urgent", "must_mention": ["jaundice", "pancreatic", "imaging", "bile"]},
]

SYSTEM_PROMPT_EVAL = """You are a medical triage AI specialized in pancreatic cancer early detection. Analyze the patient presentation and provide a structured assessment.

Your response MUST be valid JSON with this structure:
{
  "risk_assessment": "low|medium|high|critical",
  "conditions": [{"name": "condition", "likelihood": "low|medium|high", "reasoning": "..."}],
  "urgency": "self-care|routine|urgent|emergency",
  "recommended_actions": ["action 1", "action 2"],
  "reasoning_chain": "Detailed clinical reasoning explaining your assessment"
}

Key guidelines:
- Always consider pancreatic cancer in patients over 50
- New-onset diabetes after 50 with weight loss or back pain = high suspicion
- Painless jaundice = emergency until proven otherwise
- BRCA2/Lynch carriers have elevated PC risk
- Match urgency to clinical severity"""

def extract_json(text):
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        try: return json.loads(text)
        except: pass
    depth, start = 0, -1
    for i, c in enumerate(text):
        if c == "{":
            if start == -1: start = i
            depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0 and start != -1:
                try: return json.loads(text[start:i+1])
                except: start = -1
    return {}

def check_term(text, term):
    text_lower = text.lower()
    synonyms = {
        "pancreatic": ["pancreatic"],
        "family history": ["family history", "familial", "hereditary"],
        "CT": ["ct scan", "ct", "computed tomography"],
        "imaging": ["imaging", "ct", "mri", "ultrasound", "scan"],
        "weight": ["weight", "pound", "kg", "lbs"],
        "IBS": ["ibs", "irritable bowel"],
        "Trousseau": ["trousseau", "hypercoagulable", "clot"],
        "diabetes": ["diabetes", "diabetic"],
        "jaundice": ["jaundice", "yellow", "icterus"],
        "bile": ["bile", "biliary", "obstructive"],
        "urgent": ["urgent", "emergency", "immediate"],
    }
    if term in synonyms:
        return any(p in text_lower for p in synonyms[term])
    return term.lower() in text_lower

clinical_depth_words = ["risk", "symptom", "age", "pattern", "imaging", "suggest",
                       "recommend", "diagnosis", "history", "guideline", "evidence"]

def evaluate_one(response, s):
    result = {"scenario": s["name"], "passed": True, "issues": [], "scores": {}}
    extracted = extract_json(response)

    actual_risk = extracted.get("risk_assessment", "").lower()
    actual_urgency = extracted.get("urgency", "").lower()
    reasoning = extracted.get("reasoning_chain", "")

    risk_order = ["low", "medium", "high", "critical"]
    expected_idx = risk_order.index(s["expected_risk"]) if s["expected_risk"] in risk_order else -1
    actual_idx = risk_order.index(actual_risk) if actual_risk in risk_order else -1
    if actual_risk == s["expected_risk"]:
        result["scores"]["risk"] = 1.0
    elif actual_idx >= 0 and expected_idx >= 0 and abs(actual_idx - expected_idx) == 1:
        result["scores"]["risk"] = 0.5
        result["issues"].append(f"Risk off by 1: expected {s['expected_risk']}, got {actual_risk}")
    else:
        result["scores"]["risk"] = 0.0
        result["issues"].append(f"Risk wrong: expected {s['expected_risk']}, got {actual_risk}")
        result["passed"] = False

    if actual_urgency == s["expected_urgency"]:
        result["scores"]["urgency"] = 1.0
    elif s["expected_urgency"] in ("urgent", "emergency") and actual_urgency in ("urgent", "emergency"):
        result["scores"]["urgency"] = 0.7
    elif s["expected_urgency"] == "routine" and actual_urgency in ("self-care", "routine"):
        result["scores"]["urgency"] = 0.7
    else:
        result["scores"]["urgency"] = 0.0
        result["issues"].append(f"Urgency wrong: expected {s['expected_urgency']}, got {actual_urgency}")

    depth_score = sum(1 for w in clinical_depth_words if w in reasoning.lower())
    result["scores"]["reasoning"] = min(1.0, depth_score / 5)

    if s.get("must_mention"):
        found = sum(1 for t in s["must_mention"] if check_term(response, t))
        missing = [t for t in s["must_mention"] if not check_term(response, t)]
        result["scores"]["terms"] = found / len(s["must_mention"])
        for m in missing:
            result["issues"].append(f"Missing term: {m}")
        if result["scores"]["terms"] < 0.4:
            result["passed"] = False

    for t in s.get("should_not_mention", []):
        if check_term(response, t):
            result["issues"].append(f"Should not mention: {t}")
            result["passed"] = False

    scores = [v for k, v in result["scores"].items()]
    result["overall"] = sum(scores) / len(scores) if scores else 0.0
    return result


FastLanguageModel.for_inference(model)
all_results = []

for s in EVAL_SCENARIOS:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT_EVAL}]},
        {"role": "user", "content": [{"type": "text", "text": f"Please assess my symptoms:\n\n{s['input']}"}]}
    ]
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=input_ids, max_new_tokens=512, temperature=0.3, top_p=0.9, top_k=40)
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)

    r = evaluate_one(response, s)
    all_results.append(r)

    icon = "PASS" if r["passed"] else "FAIL"
    print(f"[{icon}] {s['name']}: {r['overall']:.2f}")
    for issue in r["issues"]:
        print(f"       {issue}")

print(f"\n{'='*60}")
print("RESULTS")
print('=' * 60)
passed = sum(1 for r in all_results if r['passed'])
print(f"Passed: {passed}/{len(all_results)} ({100*passed/len(all_results):.0f}%)")
avg = {}
for r in all_results:
    for k, v in r['scores'].items():
        avg.setdefault(k, []).append(v)
for metric, scores in sorted(avg.items()):
    print(f"  {metric}: {sum(scores)/len(scores):.2f}")

## Results

After 3 epochs of training on 280 examples, here is what the model achieves on 10 held-out clinical scenarios:

| Metric | Score |
|--------|-------|
| Pass rate | 8/10 (80%) |
| Risk assessment | 0.80 |
| Urgency triage | 0.81 |
| Clinical terms | 1.00 |
| Reasoning depth | 0.60 |
| **Overall** | **0.82** |

The model correctly flags high-risk presentations (new-onset diabetes + weight loss, painless jaundice, BRCA2 carriers) while avoiding over-escalation on benign cases (musculoskeletal pain, IBS in young patients).

### Links
- [Kaggle Notebook](https://www.kaggle.com/code/adhyaaykarnwal/clover-cancer-gemma-4-fine-tuning)
- [Dataset](https://www.kaggle.com/datasets/adhyaaykarnwal/clover-cancer-data)
- [LoRA Weights](https://www.kaggle.com/datasets/adhyaaykarnwal/clover-cancer-lora)
- [GitHub](https://github.com/adhyaayk/clover-cancer)

Authored by Adhyaay Karnwal